**IN:** directory of pdfs we want to chunk and vectorize

**OUT:** chromadb vector databases for each paper, named by their ID

format:
- chromadb
  - BO1005
    - [collection "BO1005"]
  - JO1013
    - [collection "JO1013"]
  - ...



In [1]:
# Import libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import chromadb
import os
from chromadb.utils import embedding_functions

/Users/pk_3/miniconda3/envs/alb/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

CHROMA_PATH = "/Users/pk_3/My_Documents/CTC/Alliance-Bioversity-CIAT/chroma_db"
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_or_create_collection(
    name="papers",
    embedding_function=embedding_fn
)

pdf_files = [f for f in os.listdir("pdfs") if f.endswith('.pdf')]

for pdf_file in pdf_files:
    PAPER_ID = pdf_file.split('-')[0]
    PDF_PATH = f"pdfs/{pdf_file}"
    CHROMA_PATH = f"chroma_db/{PAPER_ID}"

    loader = PyPDFLoader(PDF_PATH)
    raw_documents = loader.load()

    # chunking the document
    # TODO see if we should edit these parameters and if so test out different options
    text_splitter = RecursiveCharacterTextSplitter( 
        chunk_size=500,
        chunk_overlap=100,
        length_function=len,
        is_separator_regex=False,
        # TODO add explicit embedding function (https://docs.trychroma.com/docs/embeddings/embedding-functions)
    )
    chunks = text_splitter.split_documents(raw_documents)

    documents = []
    ids = []
    metadatas = []

    for i, chunk in enumerate(chunks):
        documents.append(chunk.page_content)
        ids.append(f"{PAPER_ID}_chunk_{i}")
        metadatas.append({"paper_id": PAPER_ID})   # FIXED: metadata added

    # TODO add custom table chunks to collection
    
    # Adding to chromadb

    collection.upsert(
        documents=documents,
        ids=ids,
        metadatas=metadatas
    )

ValueError: Non-empty lists are required for ['ids', 'documents'] in upsert.

In [ ]:
# TODO 
# get chunks of specific tables in a better format such as a md table

# pip install pdfplumber pymupdf openai
from pathlib import Path
import pdfplumber
import re

try:
    import fitz  # PyMuPDF (optional, for image export)
except ImportError:
    fitz = None

from openai import OpenAI

MODEL = "gpt-5-nano"
client = OpenAI()  # make sure OPENAI_API_KEY is set in your environment


# ---------------- PDF -> Markdown ----------------

def _rows_to_markdown_table(rows):
    """
    Convert a 2D list of rows to a GitHub-flavored Markdown table.
    Assumes first row is header if present; falls back to generic headers otherwise.
    """
    if not rows:
        return ""

    # Replace None with empty strings to avoid 'None' text in Markdown
    rows = [[("" if c is None else str(c)).strip() for c in r] for r in rows]

    header = rows[0]
    # If header looks empty, synthesize generic headers
    if all(cell == "" for cell in header):
        header = [f"col{i+1}" for i in range(len(header))]
        body = rows
    else:
        body = rows[1:]

    sep = ["---"] * len(header)

    lines = [
        "| " + " | ".join(header) + " |",
        "| " + " | ".join(sep) + " |",
    ]
    for r in body:
        # pad short rows
        if len(r) < len(header):
            r = r + [""] * (len(header) - len(r))
        lines.append("| " + " | ".join(r) + " |")

    return "\n".join(lines)


def export_markdown(pdf_path):
    """
    Convert a PDF to a Markdown file containing page text, extracted tables.

    Returns the Path to the written Markdown file.
    """
    pdf_path = Path(pdf_path).expanduser()
    outdir = pdf_path.with_suffix("")   # e.g. bo1005-leketa-2019/
    outdir.mkdir(exist_ok=True)

    md_parts = [f"# Extracted content: {pdf_path.name}\n"]

    # ---- Text + Tables ----
    with pdfplumber.open(pdf_path) as pdf:
        for pnum, page in enumerate(pdf.pages, start=1):
            md_parts.append(f"\n---\n\n## Page {pnum}\n")

            text = page.extract_text() or ""
            if text.strip():
                md_parts.append("### Text\n")
                # Use fenced block to preserve layout-ish formatting
                md_parts.append("```\n" + text + "\n```\n")

            tables = page.extract_tables() or []
            for tnum, table in enumerate(tables, start=1):
                md_parts.append(f"\n### Table {tnum}\n")
                md_parts.append(_rows_to_markdown_table(table) + "\n")

    md_text = "\n".join(md_parts)
    md_path = outdir / f"{pdf_path.stem}.md"
    md_path.write_text(md_text, encoding="utf-8")

    print(f"Markdown written to: {md_path}")

    return md_path


# ---------------- Remove References/Bibliography ----------------

# def remove_references_section(text: str) -> str:
#     """
#     Remove the References / Bibliography section (and everything after it)
#     from a paper's text or Markdown.

#     Looks for common patterns like a heading 'References' or 'REFERENCES'
#     on a line by itself (including markdown-style headings).
#     """

#     # 1) Markdown headings like "## References" or "### REFERENCES"
#     pattern_md = re.compile(
#         r"\n#{1,6}\s*references\b.*$", re.IGNORECASE | re.DOTALL
#     )
#     m = pattern_md.search(text)
#     if m:
#         return text[: m.start()].rstrip()

#     # 2) Standalone REFERENCES line (all caps)
#     pattern_caps = re.compile(
#         r"\n[ \t]*REFERENCES[ \t]*\n.*$", re.DOTALL
#     )
#     m = pattern_caps.search(text)
#     if m:
#         return text[: m.start()].rstrip()

#     # 3) Standalone 'References' line (case-insensitive)
#     pattern_plain = re.compile(
#         r"\n[ \t]*References[ \t]*\n.*$", re.IGNORECASE | re.DOTALL
#     )
#     m = pattern_plain.search(text)
#     if m:
#         return text[: m.start()].rstrip()

#     # If nothing matched, return unchanged
#     return text


# ---------------- OpenAI: Markdown -> CSVs for all tables ----------------

def extract_tables_to_csv_via_api(
    markdown_text: str,
    out_dir: Path,
    base_name: str = "table",
    client: OpenAI = client,
    model: str = MODEL,
) -> list[Path]:
    """
    Use the OpenAI API on the full Markdown (from a research paper) to:
      - find all tables,
      - convert them to CSV,
      - and save each as a separate CSV file.

    The model is instructed that:
      - the input is a Markdown file (not just a table),
      - it should not worry about superscripts/footnote markers,
      - and it must output each table as its own ```csv ... ``` block.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    system_msg = (
        "You are a precise table extraction and conversion assistant. "
        "You receive the full contents of a research paper in Markdown format "
        "(including headings, plain text, code fences, and GitHub-flavored Markdown tables).\n\n"
        "Your job:\n"
        "1. Find every data table in the Markdown (e.g., sections like '### Table 1' followed "
        "   by a Markdown table).\n"
        "2. Convert each table to CSV.\n\n"
        "Output format rules:\n"
        "- For each table, output a separate fenced block using the form:\n"
        "  ```csv\n"
        "  ...CSV content...\n"
        "  ```\n"
        "- Do NOT include any commentary, explanations, or Markdown outside of these csv blocks.\n"
        "- Use commas as separators.\n"
        "- Preserve cell contents as text (not including things like '995.0a' or '865.0b' or '±'); "
        "  do NOT try to interpret superscripts, footnote markers, or significance letters.\n"
        "- Do not attempt to reconstruct missing data; if a cell is blank in the table, leave it blank in CSV."
    )

    user_msg = (
        "The following content is a Markdown file exported from a research paper. "
        "It may contain headings, normal text, code fences with raw page text, and one or more "
        "GitHub-flavored Markdown tables. Please find ALL tables and output them as CSV, "
        "following the rules above.\n\n"
        "Here is the Markdown:\n\n"
        f"{markdown_text}"
    )

    # NOTE: no temperature argument here, since gpt-5-nano only supports the default.
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )

    content = resp.choices[0].message.content or ""
    # Extract all ```csv ... ``` blocks
    csv_blocks = re.findall(r"```csv(.*?)```", content, flags=re.DOTALL | re.IGNORECASE)

    csv_paths: list[Path] = []
    for idx, block in enumerate(csv_blocks, start=1):
        csv_text = block.strip()
        out_path = out_dir / f"{base_name}-table{idx}.csv"
        out_path.write_text(csv_text, encoding="utf-8")
        csv_paths.append(out_path)

    return csv_paths



# ---------------- End-to-end helper ----------------

def process_pdf_to_csv_tables(pdf_path: str | Path) -> list[Path]:
    """
    End-to-end:
      1. Convert PDF to Markdown.
      2. Remove References/Bibliography section from the Markdown.
      3. Use OpenAI API to find all tables and write them as CSV files.

    Returns list of Paths to the CSV files.
    """
    pdf_path = Path(pdf_path)
    md_path = export_markdown(pdf_path)

    md_text = md_path.read_text(encoding="utf-8")
    md_text_no_refs = md_text #remove_references_section(md_text)

    out_dir = md_path.parent / "tables"
    csv_paths = extract_tables_to_csv_via_api(
        markdown_text=md_text_no_refs,
        out_dir=out_dir,
        base_name=pdf_path.stem,
    )
    print("CSV tables written to:")
    for p in csv_paths:
        print("  -", p)
    return csv_paths


# ---------------- Example usage ----------------

if __name__ == "__main__":
    process_pdf_to_csv_tables(
        "pdfs/bo1005-leketa-2019.pdf"
    )
